In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

C:\Users\ARKAN\AppData\Local\Temp\ipykernel_8516\3309110397.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [8]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [11]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)



In [9]:
# Load documents from the directory
from langchain_community.document_loaders import DirectoryLoader

dict_loader = DirectoryLoader(r"E:\RAG\Data\New folder", glob="*.txt", loader_cls=TextLoader)

dict_docs = dict_loader.load()

print(f"Loaded {len(dict_docs)} documents")
print(f"\nFirst document preview:")
print(dict_docs[0].page_content[:200] + "...")

Loaded 3 documents

First document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


In [10]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,separators=["\n\n", "\n", " ", ""])
chunks = text_splitter.split_documents(dict_docs)

print(f"Created {len(chunks)} chunks from {len(dict_docs)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 7 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals...
Metadata: {'source': 'E:\\RAG\\Data\\New folder\\doc_0.txt'}


In [25]:
chunks[1]

Document(metadata={'source': 'E:\\RAG\\Data\\New folder\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through')

In [11]:
# Embedding model
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))



In [12]:
## create chomadb vector store
from langchain_community.vectorstores import Chroma
persistent_directory = "./chroma_db"

## create vector store with open ai embeddings
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persistent_directory,
    collection_name="my_collection"
)

print(f"Vector store created with {vector_store._collection.count()} vectors")
print(f"Persisted to: {persistent_directory}")

Vector store created with 14 vectors
Persisted to: ./chroma_db


In [40]:
## Text Similarity Search
query = "What is NLP?"

output = vector_store.similarity_search(query, k=3)
print(f"\nTop 3 similar documents for query: '{query}'")
for i, doc in enumerate(output):    
    print(f"\nDocument {i+1}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")


Top 3 similar documents for query: 'What is NLP?'

Document 1:
Content: Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
Metadata: {'source': 'E:\\RAG\\Data\\New folder\\doc_2.txt'}

Document 2:
Content: Deep Learning and Neural Networks...
Metadata: {'source': 'E:\\RAG\\Data\\New folder\\doc_1.txt'}

Document 3:
Content: Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning...
Metadata: {'source': 'E:\\RAG\\Data\\New folder\\doc_1.txt'}


In [41]:
## Advanced Similarity Search with score

result_with_score = vector_store.similarity_search_with_score(query, k=3)
print(f"\nTop 3 similar documents with scores for query: '{query}'")   
for i, (doc, score) in enumerate(result_with_score):
    print(f"\nDocument {i+1}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")
    print(f"Similarity Score: {score}")


Top 3 similar documents with scores for query: 'What is NLP?'

Document 1:
Content: Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
Metadata: {'source': 'E:\\RAG\\Data\\New folder\\doc_2.txt'}
Similarity Score: 0.6391320824623108

Document 2:
Content: Deep Learning and Neural Networks...
Metadata: {'source': 'E:\\RAG\\Data\\New folder\\doc_1.txt'}
Similarity Score: 1.181547999382019

Document 3:
Content: Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning...
Metadata: {'source': 'E:\\RAG\\Data\\New folder\\doc_1.txt'}
Similarity Score: 1.2523789405822754


# Modern RAG Chain

In [4]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [110]:
import langchain
print(langchain.__version__)

1.3.11


In [28]:
# convert vector store to retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000019E592EDAD0>, search_kwargs={'k': 3})

In [37]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know even user asked question multiple times. 
Use three sentences maximum and keep the answer concise.

Context: {context} """

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [38]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know even user asked question multiple times. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context} "), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [26]:
# Method -1: Initialize the ChatOpenAI model with the specified parameters
from langchain_openrouter import ChatOpenRouter # here you can use ChatOpenRouter instead of ChatOpenAI

model = ChatOpenRouter(
    model="openai/gpt-oss-120b",
    temperature=0.8,
)




In [80]:
# Method -1: Initialize the ChatOpenAI model with the specified parameters
from langchain.chat_models.base import init_chat_model
llm = init_chat_model("openai:gpt-oss-120b", temperature=0.7, max_tokens=500, api_key=os.getenv("OPENROUTER_API_KEY"))

In [39]:
documents_chain = create_stuff_documents_chain(llm=model, prompt=prompt)
documents_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know even user asked question multiple times. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context} "), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 

In [116]:
# create Final Rag chain
rag_chain = create_retrieval_chain(
    retriever,documents_chain
)

In [117]:
response = rag_chain.invoke({"input": "What is GPU?"})
response['answer']

'I don’t know.'

In [76]:
response

{'input': 'What is GPU?',
 'context': [Document(metadata={'source': 'E:\\RAG\\Data\\New folder\\doc_1.txt'}, page_content='excel at sequential data processing.'),
  Document(metadata={'source': 'E:\\RAG\\Data\\New folder\\doc_1.txt'}, page_content='Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  Document(metadata={'source': 'E:\\RAG\\Data\\New folder\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recogni

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [118]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [119]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [120]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [123]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | model
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000021E46B70950>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openrouter': '0.2.5'}}, output_version=None, profile={'name': 'gpt-oss-120b', 'release_date': '2025-08-05', '

In [124]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

'Deep Learning is a **subset of machine learning** that relies on **artificial neural networks**. These networks are **inspired by the human brain** and are built from **layers of interconnected nodes** (neurons). By stacking many such layers, deep learning models can automatically learn hierarchical representations of data, which has driven breakthroughs in areas such as computer vision, natural language processing, and speech recognition.'

### Add New Document in Existing Vector Store

In [1]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [2]:
# Create a new Document object for the new document
from langchain_core.documents import Document
new_doc = Document(page_content=new_document, metadata={"source": "reinforcement_learning.txt"})

In [13]:
# Split the new Document into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,separators=["\n\n", "\n", " ", ""])
new_chunks = text_splitter.split_documents([new_doc])


# Add the new chunks to the existing vector store
vector_store.add_documents(new_chunks)

['bc2c5a73-2bd0-4803-86c6-a35519c98c68',
 '16222125-938c-40db-8053-3d419dbaf765',
 'e3155f80-b46b-4421-8145-b6c4f6c2e6a8']

## Advanced RAG Technique - Conversational memory

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate

In [24]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")

])

In [30]:
# create history aware retriever 
history_aware_retriever = create_history_aware_retriever(
    model, retriever, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000019E592EDAD0>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(ta

In [57]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(model, qa_prompt)

# create conversation RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)

In [58]:
chat_history = []
# First Question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is the machine learning?"
})

print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that lets systems learn and improve from experience without being explicitly programmed. It encompasses methods that enable computers to recognize patterns, make predictions, or take actions based on data. The main approaches include supervised, unsupervised, and reinforcement learning.


In [59]:
result1

{'chat_history': [],
 'input': 'What is the machine learning?',
 'context': [Document(metadata={'source': 'E:\\RAG\\Data\\New folder\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
  Document(metadata={'source': 'E:\\RAG\\Data\\New folder\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised 

In [60]:
chat_history.extend([
    HumanMessage(content=result1['input']),
    AIMessage(content=result1['answer'])
])
chat_history

[HumanMessage(content='What is the machine learning?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a subset of artificial intelligence that lets systems learn and improve from experience without being explicitly programmed. It encompasses methods that enable computers to recognize patterns, make predictions, or take actions based on data. The main approaches include supervised, unsupervised, and reinforcement learning.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [61]:
# Second Question
result2 = conversational_rag_chain.invoke(
    {
        "input": "Explain their types?",
        "chat_history": chat_history
    }
)

print(f"Q: Explain their types?")
print(f"A: {result2['answer']}")

Q: Explain their types?
A: - **Supervised learning** trains a model on labeled data, letting it predict outputs (e.g., classifications or numeric values) for new inputs.  
- **Unsupervised learning** works with unlabeled data, discovering hidden structures such as clusters or low‑dimensional representations.  
- **Reinforcement learning** teaches an agent to make sequential decisions by rewarding desirable actions and penalizing undesirable ones, learning optimal policies through trial‑and‑error.


In [48]:
result2

{'input': 'Explain their types?',
 'chat_history': [HumanMessage(content='What is the machine learning?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a subset of artificial intelligence that lets systems learn and improve from experience without being explicitly programmed. It encompasses three main types—supervised, unsupervised, and reinforcement learning—each using different ways to train models.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is the machine learning?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a subset of artificial intelligence that lets systems learn and improve from experience without being explicitly programmed. It encompasses three main types—supervised, unsupervised, and reinforcement learning—each using different ways to train models.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_c

## RAG System with FAISS VectorStore

In [1]:
import os
from dotenv import load_dotenv

#load document
from langchain_core.documents import Document

#splitting and embedding
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

#Vector store
from langchain_classic.vectorstores import FAISS

#chains
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

#prompt templates
from langchain_core.prompts import ChatPromptTemplate

#helping library
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

# load the enviroment
load_dotenv()

e:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [5]:
# text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=[" "],
    length_function=len
)

chunks = text_splitter.split_documents(sample_documents)

In [6]:

print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 4 chunks from 4 documents

Example chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [15]:
#load the embedding model 
load_dotenv()
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))


In [16]:
# create FAISS Vectore store
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Vector store created with 4 vectors


In [17]:
vectors = embeddings.embed_documents([
    "Hello",
    "How are you?"
])

print(len(vectors))

2


In [18]:
query="what is NLP?"
vectorstore.similarity_search(query,k=3)

[Document(id='f8bfceea-fefe-42a8-9ba7-ee9c3720f7fa', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.\n        It combines computational linguistics with machine learning and deep learning models.\n        Applications include chatbots, translation, sentiment analysis, and text summarization.'),
 Document(id='c86bc744-606c-4278-8b03-ab47b5643f63', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'),
 Document(id='89ebe446-abed-4439-b4a0-6ae1a9bbb0c4', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enable

In [29]:
# Build RAG Chain with LCEL

from langchain.chat_models import init_chat_model

os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
chat_model = init_chat_model(
    model="google/gemma-4-31b-it:free",
    base_url="https://openrouter.ai/api/v1",
    model_provider="openrouter"
    
)

In [33]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [35]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [34]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [38]:
from langchain_core.runnables import RunnablePassthrough
simple_rag_chai = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | chat_model
    | StrOutputParser
)

In [57]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1. Prompt to rephrase follow-up questions into standalone questions
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question, "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [58]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | chat_model
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [54]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [60]:
## Conversational example
print("Conversational RAG Example:")
chat_history = []

# First question
q1 = "What is machine learning?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {a1}")

Conversational RAG Example:


TooManyRequestsResponseError: Provider returned error

In [ ]:
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])


In [47]:
chat_history

[HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Based on the provided text, machine learning is a subset of AI that enables systems to learn from data. Rather than being explicitly programmed, ML algorithms find patterns in data. Common types of machine learning include supervised, unsupervised, and reinforcement learning.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [49]:
# Follow up question
q2 = "Explain their types?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history

})

print(f"Q1: {q2}")
print(f"A1: {a2}")

Q1: Explain their types?
A1: Based on the provided documents, the types for each category are as follows:

*   **Artificial Intelligence (AI):** Can be categorized into narrow AI and general AI.
*   **Machine Learning (ML):** Common types include supervised, unsupervised, and reinforcement learning.
*   **Deep Learning:** This is a subset of machine learning based on artificial neural networks that uses multiple layers to extract features from raw input.
